In [4]:
import pandas as pd
import glob
import os
import ast

file = "all-MCU2.csv"

og_df = pd.read_csv(file)
df = og_df

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ==========================================
# 1. SETUP & DATA
# ==========================================
if 'df' not in locals():
    np.random.seed(42)
    models = ['gpt-4o', 'gpt-4o-mini', 'qwen32', 'qwen14', 'phi4', 'codestral', 'codestral-p']
    processors = ['dp', 'mc', 'sg']
    statuses = ['success', 'failure']

    data = []
    for _ in range(300):
        data.append({
            'processor': np.random.choice(processors),
            'model': np.random.choice(models),
            'status': np.random.choice(statuses, p=[0.8, 0.2]),
            'latency': np.random.lognormal(0, 0.5),
            'total_tokens': np.random.randint(100, 5000)
        })
    df = pd.DataFrame(data)

for col in ['processor', 'model', 'status']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()

# ==========================================
# 2. CONFIGURATION
# ==========================================
model_map = {
    'gpt-4o': 'G4o', 'gpt-4o-mini': 'G4o-m', 'qwen32': 'Qw32B', 'qwen14': 'Qw14B',
    'phi4': 'Phi4', 'codestral': 'Co22B', 'codestral-p': 'Co22B-p'
}
proc_titles = {
    'dp': 'Data Processing', 'mc': 'Model Conversion', 'sg': 'Arduino Sketch Generation'
}

proc_order = ['dp', 'mc', 'sg']
model_order = list(model_map.values())
palette = {'success': '#006837', 'failure': '#D73027'}

df['model_display'] = df['model'].map(model_map)

metrics = [
    {'col': 'latency', 'label': 'Execution Time (s)', 'fmt': None},
    {'col': 'total_tokens', 'label': '# Tokens (Total)', 'fmt': lambda x, p: f'{int(x/1000)}k' if x >= 1000 else f'{int(x)}'}
]

# ==========================================
# 3. PLOTTING (VIOLIN)
# ==========================================
sns.set_style('white')
plt.rcParams['font.family'] = 'sans-serif'

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 7))
plt.subplots_adjust(top=0.90, hspace=0.13, wspace=0.10)

for col_idx, proc in enumerate(proc_order):
    subset = df[df['processor'] == proc]

    for row_idx, metric in enumerate(metrics):
        ax = axes[row_idx, col_idx]

        if subset.empty:
            ax.set_visible(False)
            continue

        sns.violinplot(
            data=subset,
            x='model_display',
            y=metric['col'],
            hue='status',
            hue_order=['success', 'failure'],
            order=model_order,
            split=True,
            inner='quartile',
            cut=0,
            bw_adjust=0.8,
            linewidth=0.9,
            palette=palette,
            ax=ax
        )

        if ax.get_legend():
            ax.get_legend().remove()
        ax.set_xlabel('')
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylabel(metric['label'] if col_idx == 0 else '', fontweight='bold', fontsize=13)

        if row_idx == 0:
            ax.set_title(proc_titles.get(proc, proc), fontsize=14, weight='bold', pad=5)
        if metric['fmt']:
            ax.yaxis.set_major_formatter(FuncFormatter(metric['fmt']))

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles[:2], ['Successful runs', 'Failed runs'], loc='upper center', ncol=2,
           bbox_to_anchor=(0.5, 1.01), fontsize=13, frameon=True)

plt.savefig('figs/violin_MCU.pdf', dpi=300, bbox_inches='tight')
plt.show()